# Generate cell-specific GRNs with scCAFM SFM

This notebook shows how to use the **Structure Foundation Module (SFM)** part of scCAFM to generate cell-specific gene regulatory networks (GRNs) from single-cell RNA-seq data.

scCAFM is designed as a modular project. This notebook covers only SFM, which models context-specific regulatory structure. Future scCAFM tutorials may cover EFM separately.

## 1. Environment

Run this notebook from the scCAFM repository root, or from `docs/`. If scCAFM is not installed yet, install it from the local checkout and download the model assets first.

```bash
pip install .
hf download kaichenxu/scCAFM --local-dir assets
```

If you use the pinned Python 3.12 environment from this repo, install with:

```bash
pip install .[py312]
```

In [ ]:
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
import sys

import anndata as ad
import pandas as pd
import torch
from torch.utils.data import DataLoader, SequentialSampler

cwd = Path.cwd().resolve()
if (cwd / "src").exists() and (cwd / "configs").exists():
    repo_root = cwd
elif (cwd.parent / "src").exists() and (cwd.parent / "configs").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Run this notebook from the scCAFM repository root or from docs/.")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repository root: {repo_root}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configure inputs

Set `input_h5ad` to your single-cell dataset. The dataset should contain genes in `adata.var_names` or in the column named by `gene_key`. The default config expects species metadata in `adata.obs['species']`; if your dataset does not have that column, either add it or set `species_key = None` below to let scCAFM default to human.

`top_k_per_cell` keeps the strongest edges for each cell. `score_threshold` can be set to a float to additionally remove low-scoring edges.

In [ ]:
# Required: point this to your AnnData file.
input_h5ad = repo_root / "path/to/your_data.h5ad"

# Model assets. Use repo-local assets if downloaded, or replace with "kaichenxu/scCAFM".
model_source = repo_root / "assets"

# Optional: set to a fine-tuned SFM checkpoint file or a directory containing sfm_model.safetensors.
checkpoint_path = None

# Output edge list.
output_csv = repo_root / "results" / "sfm_grn_edges.csv"

# Inference/export controls.
batch_size = 8
top_k_per_cell = 1000
score_threshold = None

# Optional data-column overrides. Leave as USE_CONFIG to keep configs/eval_grn.yaml defaults.
# Set a key to None to intentionally disable that metadata column.
USE_CONFIG = "__use_config_default__"
gene_key = USE_CONFIG
platform_key = USE_CONFIG
species_key = USE_CONFIG
tissue_key = USE_CONFIG
disease_key = USE_CONFIG

# Optional sequence-length override. Larger values keep more genes but use more memory.
max_length = None

if not input_h5ad.exists():
    raise FileNotFoundError(
        f"Set input_h5ad to a real .h5ad file before continuing. Current value: {input_h5ad}"
    )

output_csv.parent.mkdir(parents=True, exist_ok=True)

## 3. Load SFM assets and config

This section starts from `configs/eval_grn.yaml` because that config already contains evaluation-friendly preprocessing defaults. We then point it at a single `.h5ad` file and use the SFM model package from `model_source`.

In [ ]:
from src.assets import (
    SFM_MODEL_NAME,
    apply_model_assets_to_runtime_config,
    load_model_state_dict,
    load_sfm_config,
    resolve_model_assets,
)
from src.config import load_yaml_config
from src.data import PreprocessedScDataset, PretrainingDataBundle, ScBatchCollator, build_evaluation_assets
from src.distributed import RuntimeContext, move_batch_to_device
from src.trainer.builders import build_model

def resolve_checkpoint_path(checkpoint_path, default_model_path: Path) -> Path:
    if checkpoint_path is None or str(checkpoint_path).strip() == "":
        resolved = default_model_path
    else:
        candidate = Path(checkpoint_path).expanduser().resolve()
        resolved = candidate / SFM_MODEL_NAME if candidate.is_dir() else candidate
    if not resolved.exists():
        raise FileNotFoundError(f"SFM checkpoint not found: {resolved}")
    return resolved

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime = RuntimeContext(
    rank=0,
    world_size=1,
    local_rank=0,
    device=device,
    distributed=False,
    is_main=True,
)

eval_config = load_yaml_config(repo_root / "configs" / "eval_grn.yaml")
eval_config["model_source"] = str(model_source)

data_cfg = deepcopy(eval_config.get("data", {}))
data_cfg["train_paths"] = [str(input_h5ad.resolve())]
data_cfg["batch_size"] = int(batch_size)
data_cfg["num_workers"] = 0
data_cfg["pin_memory"] = bool(device.type == "cuda")
data_cfg["condition_vocab"] = {"regenerate": False}
data_cfg["condition_mask"] = {"enabled": False, "unk_ratio": 0.0}

for key, value in {
    "gene_key": gene_key,
    "platform_key": platform_key,
    "species_key": species_key,
    "tissue_key": tissue_key,
    "disease_key": disease_key,
    "max_length": max_length,
}.items():
    if value != "__use_config_default__":
        data_cfg[key] = value

eval_config["data"] = data_cfg
eval_config.setdefault("evaluator", {})["foundation_name"] = "sfm"
if checkpoint_path is not None:
    eval_config["evaluator"]["checkpoint_path"] = str(checkpoint_path)

assets = resolve_model_assets(
    model_source=eval_config["model_source"],
    require_model_weights=checkpoint_path is None,
)
sfm_config = load_sfm_config(assets.sfm_config)
config = apply_model_assets_to_runtime_config(
    eval_config,
    assets,
    require_model_weights=checkpoint_path is None,
)
checkpoint_file = resolve_checkpoint_path(
    config.get("evaluator", {}).get("checkpoint_path"),
    assets.sfm_model,
)

print(f"Device: {device}")
print(f"Assets: {assets.local_dir}")
print(f"Checkpoint: {checkpoint_file}")

## 4. Build the tokenizer, dataset, and SFM model

The notebook uses the same tokenizer, preprocessing rules, vocabulary, TF lists, and checkpoint-loading utilities used by the command-line evaluator. Cells are processed in sequential order so the exported `cell_index` matches the processed AnnData row order.

In [ ]:
data_assets = build_evaluation_assets(config=config, runtime=runtime)

raw_adata = ad.read_h5ad(input_h5ad)
processed_adata = data_assets.preprocessor(raw_adata) if data_assets.preprocessor is not None else raw_adata
processed_cell_count = int(processed_adata.n_obs)
processed_gene_count = int(processed_adata.n_vars)

dataset = PreprocessedScDataset(
    adata=processed_adata,
    tokenizer=data_assets.tokenizer,
    gene_key=data_assets.gene_key,
    preprocessor=None,
)
loader = DataLoader(
    dataset,
    batch_size=int(batch_size),
    sampler=SequentialSampler(dataset),
    shuffle=False,
    collate_fn=ScBatchCollator(),
    num_workers=0,
    pin_memory=bool(device.type == "cuda"),
)

model_bundle = PretrainingDataBundle(
    train_loader=None,
    train_sampler=None,
    token_dict=data_assets.token_dict,
    cond_vocab_size=data_assets.cond_vocab_size,
    train_size=len(dataset),
    path=input_h5ad.resolve(),
)
model = build_model(
    sfm_config=sfm_config,
    data_bundle=model_bundle,
    assets=assets,
    runtime_config=config.get("runtime", {}),
)
model.load_state_dict(load_model_state_dict(checkpoint_file))
model.to(device)
model.eval()

print(f"Raw cells: {raw_adata.n_obs}; processed cells: {processed_cell_count}")
print(f"Raw genes: {raw_adata.n_vars}; processed genes: {processed_gene_count}")
print(f"Batches: {len(loader)}")

## 5. Export cell-specific GRN edges

SFM returns a dense GRN score tensor with shape `(cells, genes, genes)` for each batch. A score at `[cell, source, target]` is the directed regulatory confidence for `source -> target` in that cell.

The export below keeps only valid source and target genes, removes self-loops, restricts source genes to TFs, and applies `score_threshold` and `top_k_per_cell`.

In [ ]:
def build_token_id_to_gene(token_dict: pd.DataFrame) -> dict[int, str]:
    mapping: dict[int, str] = {}
    for _, row in token_dict.iterrows():
        token_id = int(row["token_index"])
        symbol = row.get("gene_symbol")
        gene_id = row.get("gene_id")
        if pd.notna(symbol) and str(symbol).strip():
            mapping[token_id] = str(symbol)
        elif pd.notna(gene_id) and str(gene_id).strip():
            mapping[token_id] = str(gene_id)
        else:
            mapping[token_id] = str(token_id)
    return mapping

def edges_from_batch(
    grn: torch.Tensor,
    tokens: dict[str, torch.Tensor | None],
    *,
    cell_offset: int,
    token_id_to_gene: dict[int, str],
    top_k_per_cell: int | None,
    score_threshold: float | None,
) -> list[dict[str, object]]:
    input_ids = tokens["input_ids"].detach().cpu().to(torch.long)
    non_tf_mask = tokens["non_tf_mask"].detach().cpu().to(torch.bool)
    padding_mask = tokens.get("padding_mask")
    if padding_mask is None:
        padding_mask_cpu = torch.zeros_like(non_tf_mask, dtype=torch.bool)
    else:
        padding_mask_cpu = padding_mask.detach().cpu().to(torch.bool)

    grn_cpu = grn.detach().cpu().to(torch.float32)
    rows: list[dict[str, object]] = []

    for batch_cell_idx in range(grn_cpu.shape[0]):
        active_mask = ~padding_mask_cpu[batch_cell_idx]
        source_mask = active_mask & ~non_tf_mask[batch_cell_idx]
        target_mask = active_mask
        candidate_mask = source_mask[:, None] & target_mask[None, :]
        candidate_mask.fill_diagonal_(False)

        src_pos, tgt_pos = candidate_mask.nonzero(as_tuple=True)
        if src_pos.numel() == 0:
            continue

        scores = grn_cpu[batch_cell_idx, src_pos, tgt_pos]
        if score_threshold is not None:
            keep = scores >= float(score_threshold)
            src_pos = src_pos[keep]
            tgt_pos = tgt_pos[keep]
            scores = scores[keep]

        if scores.numel() == 0:
            continue

        if top_k_per_cell is not None and int(top_k_per_cell) > 0 and scores.numel() > int(top_k_per_cell):
            top_indices = torch.topk(scores, k=int(top_k_per_cell), largest=True).indices
            src_pos = src_pos[top_indices]
            tgt_pos = tgt_pos[top_indices]
            scores = scores[top_indices]

        order = torch.argsort(scores, descending=True)
        src_pos = src_pos[order]
        tgt_pos = tgt_pos[order]
        scores = scores[order]

        cell_index = cell_offset + batch_cell_idx
        for src_i, tgt_i, score in zip(src_pos.tolist(), tgt_pos.tolist(), scores.tolist()):
            source_token_id = int(input_ids[batch_cell_idx, src_i].item())
            target_token_id = int(input_ids[batch_cell_idx, tgt_i].item())
            rows.append(
                {
                    "cell_index": cell_index,
                    "source_gene": token_id_to_gene.get(source_token_id, str(source_token_id)),
                    "target_gene": token_id_to_gene.get(target_token_id, str(target_token_id)),
                    "score": float(score),
                    "source_token_id": source_token_id,
                    "target_token_id": target_token_id,
                }
            )
    return rows

token_id_to_gene = build_token_id_to_gene(data_assets.token_dict)
all_rows: list[dict[str, object]] = []
cell_offset = 0

with torch.no_grad():
    for batch in loader:
        tokens = move_batch_to_device(batch, device)
        model_output = model(
            tokens,
            compute_grn={"sfm": True},
            return_factors=False,
        )
        grn = model_output.foundations["sfm"].grn
        if grn is None:
            raise RuntimeError("SFM did not return a GRN tensor.")

        all_rows.extend(
            edges_from_batch(
                grn,
                tokens,
                cell_offset=cell_offset,
                token_id_to_gene=token_id_to_gene,
                top_k_per_cell=top_k_per_cell,
                score_threshold=score_threshold,
            )
        )
        cell_offset += int(tokens["input_ids"].shape[0])

edges_df = pd.DataFrame(
    all_rows,
    columns=[
        "cell_index",
        "source_gene",
        "target_gene",
        "score",
        "source_token_id",
        "target_token_id",
    ],
)
edges_df.to_csv(output_csv, index=False)

print(f"Saved {len(edges_df):,} SFM GRN edges to {output_csv}")
edges_df.head()

## 6. Evaluate exported GRN edges

If you have a reference GRN file, set `reference_grn_csv` below. The reference CSV must contain `Gene1` and `Gene2` columns, where `Gene1 -> Gene2` is a directed regulatory edge.

This section evaluates the exported edge list. For full candidate-universe benchmarking against reference GRNs, use the command-line evaluator in the next section.

In [ ]:
from src.evaluator.metrics import summarize_binary_metrics

# Optional: set this to a reference GRN CSV with Gene1 and Gene2 columns.
reference_grn_csv = None
evaluation_output_csv = repo_root / "results" / "sfm_grn_edge_evaluation.csv"

def normalize_gene_name(value: object) -> str:
    name = str(value).strip().upper()
    if name.startswith("ENSG"):
        name = name.split(".", 1)[0]
    return name

def load_reference_edges(path: str | Path) -> set[tuple[str, str]]:
    reference_df = pd.read_csv(Path(path).expanduser().resolve())
    required_columns = {"Gene1", "Gene2"}
    if not required_columns.issubset(reference_df.columns):
        raise ValueError(f"Reference GRN must contain columns {sorted(required_columns)}")
    return {
        (normalize_gene_name(src), normalize_gene_name(tgt))
        for src, tgt in zip(reference_df["Gene1"], reference_df["Gene2"])
        if normalize_gene_name(src) != normalize_gene_name(tgt)
    }

def label_exported_edges(edges: pd.DataFrame, reference_edges: set[tuple[str, str]]) -> pd.DataFrame:
    labeled = edges.copy()
    labeled["label"] = [
        int((normalize_gene_name(src), normalize_gene_name(tgt)) in reference_edges)
        for src, tgt in zip(labeled["source_gene"], labeled["target_gene"])
    ]
    return labeled

if reference_grn_csv is None:
    print("Set reference_grn_csv to evaluate exported SFM GRN edges against a reference network.")
else:
    reference_edges = load_reference_edges(reference_grn_csv)
    labeled_edges_df = label_exported_edges(edges_df, reference_edges)
    labeled_edges_df.to_csv(evaluation_output_csv, index=False)

    if labeled_edges_df.empty:
        raise ValueError("No exported edges are available to evaluate.")

    scores = torch.tensor(labeled_edges_df["score"].to_numpy(), dtype=torch.float64)
    labels = torch.tensor(labeled_edges_df["label"].to_numpy(), dtype=torch.float64)
    overall_metrics = summarize_binary_metrics(scores=scores, labels=labels)

    cell_rows = []
    for cell_index, cell_df in labeled_edges_df.groupby("cell_index"):
        cell_scores = torch.tensor(cell_df["score"].to_numpy(), dtype=torch.float64)
        cell_labels = torch.tensor(cell_df["label"].to_numpy(), dtype=torch.float64)
        cell_metrics = summarize_binary_metrics(scores=cell_scores, labels=cell_labels)
        cell_rows.append({"cell_index": int(cell_index), **cell_metrics})

    cell_metrics_df = pd.DataFrame(cell_rows)
    print(f"Reference edges: {len(reference_edges):,}")
    print(f"Labeled exported edges saved to {evaluation_output_csv}")
    display(pd.DataFrame([overall_metrics], index=["exported_edges"]))
    display(cell_metrics_df.head())

## 7. Command-line evaluation alternative

The Python workflow above directly exports per-cell GRN edges. scCAFM also provides command-line GRN evaluation against reference networks. These commands produce evaluation outputs such as `logs/evaluate_grn.log` and `results/summary_metrics.csv`; they do not currently export per-cell edge-list CSVs.

In [ ]:
# Run from the repository root after editing configs/eval_grn.yaml.
# !python -m src.evaluator.grn --eval-grn-config configs/eval_grn.yaml
# !bash scripts/eval_grn.sh

## 8. Notes

- `cell_index` is the row index after preprocessing. If preprocessing removes cells, this may differ from the original AnnData row number.
- Source genes are restricted to transcription factors using the SFM asset TF tables and each cell's species metadata.
- Increase `top_k_per_cell` for denser GRNs, or set `score_threshold` to focus on high-confidence edges.
- Increase `data.max_length` only if your GPU memory can support the larger dense `(genes, genes)` GRN tensor.